In [4]:
import pandas as pd

df = pd.read_csv('../data/raw/city_hour.csv')

print(df.shape)
print(df.columns.tolist())
print(df.head(10))
print(df['City'].unique())

(707875, 16)
['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']
        City             Datetime  PM2.5  PM10    NO    NO2    NOx  NH3    CO  \
0  Ahmedabad  2015-01-01 01:00:00    NaN   NaN  1.00  40.01  36.37  NaN  1.00   
1  Ahmedabad  2015-01-01 02:00:00    NaN   NaN  0.02  27.75  19.73  NaN  0.02   
2  Ahmedabad  2015-01-01 03:00:00    NaN   NaN  0.08  19.32  11.08  NaN  0.08   
3  Ahmedabad  2015-01-01 04:00:00    NaN   NaN  0.30  16.45   9.20  NaN  0.30   
4  Ahmedabad  2015-01-01 05:00:00    NaN   NaN  0.12  14.90   7.85  NaN  0.12   
5  Ahmedabad  2015-01-01 06:00:00    NaN   NaN  0.33  15.95  10.82  NaN  0.33   
6  Ahmedabad  2015-01-01 07:00:00    NaN   NaN  0.45  15.94  12.47  NaN  0.45   
7  Ahmedabad  2015-01-01 08:00:00    NaN   NaN  1.03  16.66  16.48  NaN  1.03   
8  Ahmedabad  2015-01-01 09:00:00    NaN   NaN  1.47  16.25  18.02  NaN  1.47   
9  Ahmedabad  2015-01-01 10:00:00    NaN

In [5]:
delhi_df = df[df['City'] == 'Delhi'].copy()

print("Total rows:", len(delhi_df))
print("Date range:", delhi_df['Datetime'].min(), "to", delhi_df['Datetime'].max())
print("\nMissing values per column:")
print(delhi_df.isnull().sum())
print("\nRows where AQI is not null:", delhi_df['AQI'].notna().sum())

Total rows: 48192
Date range: 2015-01-01 01:00:00 to 2020-07-01 00:00:00

Missing values per column:
City              0
Datetime          0
PM2.5           375
PM10           2421
NO              298
NO2             330
NOx              25
NH3             980
CO              364
SO2            2852
O3             2201
Benzene          38
Toluene          26
Xylene        18904
AQI             498
AQI_Bucket      498
dtype: int64

Rows where AQI is not null: 47694


In [7]:
# Step 1: Filter to Delhi only and make a clean copy
delhi_df = df[df['City'] == 'Delhi'].copy()

# Step 2: Parse datetime and sort
delhi_df['Datetime'] = pd.to_datetime(delhi_df['Datetime'])
delhi_df = delhi_df.sort_values('Datetime').reset_index(drop=True)

# Step 3: Drop Xylene - too much missing data (39%)
delhi_df = delhi_df.drop(columns=['Xylene', 'AQI_Bucket'])

# Step 4: Forward fill missing values (max 3 hours gap)
# This is safe for air quality data - pollution levels don't jump instantly
pollutant_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 
                  'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene']

delhi_df[pollutant_cols] = delhi_df[pollutant_cols].ffill(limit=3)

# Step 5: For AQI specifically - drop rows where it's still missing after ffill
delhi_df = delhi_df.dropna(subset=['AQI'])

# Step 6: Verify
print("Shape after cleaning:", delhi_df.shape)
print("\nRemaining missing values:")
print(delhi_df.isnull().sum())
print("\nDate range:", delhi_df['Datetime'].min(), "to", delhi_df['Datetime'].max())

Shape after cleaning: (47694, 14)

Remaining missing values:
City           0
Datetime       0
PM2.5         56
PM10        1897
NO            31
NO2           44
NOx            4
NH3          532
CO           160
SO2         2350
O3          1755
Benzene        7
Toluene        5
AQI            0
dtype: int64

Date range: 2015-01-01 16:00:00 to 2020-07-01 00:00:00


In [8]:
# Step 7: Fill remaining gaps using median for that hour-of-day
# More intelligent than global median - respects daily pollution cycles
delhi_df['hour'] = delhi_df['Datetime'].dt.hour

for col in pollutant_cols:
    if delhi_df[col].isnull().sum() > 0:
        hourly_median = delhi_df.groupby('hour')[col].transform('median')
        delhi_df[col] = delhi_df[col].fillna(hourly_median)

# Drop the helper column
delhi_df = delhi_df.drop(columns=['hour'])

# Step 8: Final check
print("Remaining missing values after hourly median fill:")
print(delhi_df.isnull().sum())
print("\nFinal shape:", delhi_df.shape)

# Step 9: Save cleaned data
delhi_df.to_csv('../data/processed/delhi_aqi_clean.csv', index=False)
print("\nSaved to data/processed/delhi_aqi_clean.csv")

Remaining missing values after hourly median fill:
City        0
Datetime    0
PM2.5       0
PM10        0
NO          0
NO2         0
NOx         0
NH3         0
CO          0
SO2         0
O3          0
Benzene     0
Toluene     0
AQI         0
dtype: int64

Final shape: (47694, 14)

Saved to data/processed/delhi_aqi_clean.csv
